# Retrain v4 -- snapvec-mined vs sqlite-vec-mined hard negatives

Colab companion to `experiments/retrain_v4_snapvec.py`.

**Hypothesis to test**. After the 2026-04-20 cosine-similarity probe
falsified the original "ANN noise" read, the open question is: does
training with snapvec-mined negatives produce a v4 that (a) matches
or exceeds v4 mined with sqlite-vec, and (b) shows quantization-aware
co-adaptation when deployed on snapvec at inference?

Experiment output is a 2x2 matrix per dataset:

| trained on \\ eval on | sqlite-vec | snapvec |
|---|---|---|
| baseline (v3, no retrain) | | |
| v4 mined from sqlite-vec | | |
| v4 mined from snapvec | | |

Comparisons worth watching:
- v4-snap (eval snapvec) vs v3 (eval snapvec): does co-adaptation happen?
- v4-snap (eval snapvec) vs v4-vec (eval snapvec): does mining backend matter?
- v4-vec vs v4-snap across both eval backends: is one strictly better?

Runs on T4 GPU in ~15-25 min per dataset at 2000 training queries.

In [ ]:
# Cell 1: Clone branch + install vstash (with snapvec extra) and deps.
!pip install -q 'sentence-transformers>=3' torch 'accelerate>=1.1.0'
!rm -rf /content/vstash
!git clone --branch experiments/snapvec-v3-sweeps https://github.com/stffns/vstash.git /content/vstash
%cd /content/vstash
!pip install -q -e '.[snapvec]'

In [ ]:
# Cell 2: Verify GPU + snapvec available.
import torch
print('cuda:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
import snapvec
print('snapvec:', snapvec.__version__ if hasattr(snapvec, '__version__') else 'installed')

In [ ]:
# Cell 3: Run on SciFact first (smallest corpus, fastest validation).
# max-queries=2000 -> roughly 2000 sqlite-vec pairs + 2000 snapvec pairs
# (both with mostly in-batch negatives plus explicit hard-negs where
# the miner finds them). Epochs=2, lr=3e-6, batch=64 matches the v3
# recipe in retrain_multi. chdir is defensive: Colab kernels can
# lose cwd across re-runs of individual cells after a restart.
import os
os.chdir('/content/vstash')
os.makedirs('experiments/results', exist_ok=True)
!python -m experiments.retrain_v4_snapvec \
    --dataset scifact \
    --max-queries 2000 \
    --train \
    --epochs 2 \
    --batch-size 64 \
    --lr 3e-6 \
    --output experiments/results/retrain_v4_snapvec_scifact.json \
    --models-dir /content/v4_models 2>&1 | tee /content/v4_scifact.log

In [ ]:
import os
os.chdir('/content/vstash')
# Cell 4: Same on NFCorpus. This is where v2 originally showed the
# biggest retrain gain vs base (+18%); worth re-running under snapvec
# mining to see if the signal replicates.
!python -m experiments.retrain_v4_snapvec \
    --dataset nfcorpus \
    --max-queries 2000 \
    --train \
    --epochs 2 \
    --batch-size 64 \
    --lr 3e-6 \
    --output experiments/results/retrain_v4_snapvec_nfcorpus.json \
    --models-dir /content/v4_models 2>&1 | tee /content/v4_nfcorpus.log

In [ ]:
import os
os.chdir('/content/vstash')
# Cell 5: Same on FiQA. This is where v2 regressed vs base. If the
# snapvec-mining path behaves differently than sqlite-vec-mining on
# FiQA, it will show here.
!python -m experiments.retrain_v4_snapvec \
    --dataset fiqa \
    --max-queries 2000 \
    --train \
    --epochs 2 \
    --batch-size 64 \
    --lr 3e-6 \
    --output experiments/results/retrain_v4_snapvec_fiqa.json \
    --models-dir /content/v4_models 2>&1 | tee /content/v4_fiqa.log

In [ ]:
import os
os.chdir('/content/vstash')
# Cell 6: Consolidate the per-dataset JSONs into one comparison table.
import json
from pathlib import Path

datasets = ['scifact', 'nfcorpus', 'fiqa']
rows = []
for ds in datasets:
    p = Path(f'experiments/results/retrain_v4_snapvec_{ds}.json')
    if not p.exists():
        print(f'missing: {p}')
        continue
    data = json.loads(p.read_text())
    ev = data.get('eval', {})
    base = ev.get('baseline', {})
    v4v = ev.get('v4_sqlite-vec', {})
    v4s = ev.get('v4_snapvec', {})
    rows.append({
        'dataset': ds,
        'base_vec': base.get('sqlite-vec', {}).get('ndcg_at_10'),
        'base_snap': base.get('snapvec', {}).get('ndcg_at_10'),
        'v4vec_on_vec': v4v.get('on_sqlite-vec', {}).get('ndcg_at_10'),
        'v4vec_on_snap': v4v.get('on_snapvec', {}).get('ndcg_at_10'),
        'v4snap_on_vec': v4s.get('on_sqlite-vec', {}).get('ndcg_at_10'),
        'v4snap_on_snap': v4s.get('on_snapvec', {}).get('ndcg_at_10'),
        'n_pairs_vec': v4v.get('n_pairs'),
        'n_pairs_snap': v4s.get('n_pairs'),
    })

print('| dataset | base/vec | base/snap | v4-vec/vec | v4-vec/snap | v4-snap/vec | v4-snap/snap |')
print('|---|---|---|---|---|---|---|')
for r in rows:
    print(
        f"| {r['dataset']} | {r['base_vec']:.4f} | {r['base_snap']:.4f} | "
        f"{r['v4vec_on_vec']:.4f} | {r['v4vec_on_snap']:.4f} | "
        f"{r['v4snap_on_vec']:.4f} | {r['v4snap_on_snap']:.4f} |"
    )

# Dump combined JSON for easy download.
out = Path('experiments/results/retrain_v4_snapvec_combined.json')
out.write_text(json.dumps({'runs': rows}, indent=2))
print(f'\nwrote {out}')

In [ ]:
# Cell 7 (optional): Zip result JSONs + logs for download.
!cd /content/vstash && zip -r /content/v4_snapvec_results.zip \
    experiments/results/retrain_v4_snapvec_*.json && \
    cp /content/v4_*.log /content/ 2>/dev/null || true
print('Download /content/v4_snapvec_results.zip to pull JSONs locally.')